# BreastDM Lesion Segmentation with MLSRS-SegFormer

English edition of the supplied experiment notebook. Saved outputs have been cleared to remove untranslated figure text; recorded metrics and selected English figures are provided separately.

This notebook preserves the original experimental cell order and calculations. It requires manual execution-order cleanup and the external dataset/checkpoints. It is not a verified Run All pipeline. See README for split-overlap and legacy distance-metric limitations.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("breastdm.training")

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
print("GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:
IMG_SIZE    = 256
NUM_CLASSES = 2
BATCH_SIZE  = 4


In [ ]:
def parse_image(img_path, mask_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_jpeg(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method='nearest')
    mask = tf.cast(mask, tf.float32)
    mask = tf.where(mask > 127.0, 1, 0)
    mask = tf.cast(mask, tf.int32)
    mask = tf.squeeze(mask, axis=-1)
    mask = tf.one_hot(mask, NUM_CLASSES)
    return img, mask

def augment(image, mask):

    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask  = tf.image.flip_left_right(mask)

    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask  = tf.image.flip_up_down(mask)

    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask

def get_all_files(root_dir):
    all_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith((".jpg", ".png")):
                all_files.append(os.path.join(root, file))
    return sorted(all_files)

def load_dataset(root, split="train"):
    img_dir  = os.path.join(root, "seg", split, "images")
    mask_dir = os.path.join(root, "seg", split, "labels")
    img_paths  = get_all_files(img_dir)
    mask_paths = get_all_files(mask_dir)
    print(f"[{split}] IMG: {len(img_paths)}  MASK: {len(mask_paths)}")
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if split == "train":                                          
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

ROOT     = "/kaggle/input/datasets/mehmetkaraman48/breastdm"
train_ds = load_dataset(ROOT, "train")
val_ds   = load_dataset(ROOT, "val")
test_ds  = load_dataset(ROOT, "test")
print("Train batches:", len(train_ds))

In [ ]:
def dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return 1.0 - (2.0 * intersection + 1e-6) / (union + 1e-6)


def total_loss(y_true, y_pred):
    cce = tf.keras.losses.CategoricalCrossentropy()(y_true, y_pred)
    return cce + dice_loss(y_true, y_pred)


def dice_coef(y_true, y_pred):
    yt = tf.cast(tf.argmax(y_true, axis=-1), tf.float32)
    yp = tf.cast(tf.argmax(y_pred, axis=-1), tf.float32)
    intersection = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt) + tf.reduce_sum(yp)
    return (2.0 * intersection + 1e-6) / (union + 1e-6)


def iou(y_true, y_pred):
    yt = tf.cast(tf.argmax(y_true, axis=-1), tf.float32)
    yp = tf.cast(tf.argmax(y_pred, axis=-1), tf.float32)
    intersection = tf.reduce_sum(yt * yp)
    union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - intersection
    return intersection / (union + 1e-6)


## MLSRS-SegFormer Architecture

Custom TensorFlow/Keras implementation with three main components:
1. **Adaptive Patch Weighting**: learnable channel weights shared across patch tokens.
2. **Spatial Attention Bias**: normalized pairwise Euclidean token distances, called Hausdorff bias in the experiment.
3. **Relative Positional Encoding**: learnable 2D relative position bias.

See README for implementation differences and evaluation limitations.


In [ ]:



class PatchEmbedding(layers.Layer):
    """
    Conv2D projection + learnable patch_weight + LayerNorm.
    Initialize patch_weight to ones (paper Eq. 1).
    """
    def __init__(self, patch_size, stride, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.stride     = stride
        self.embed_dim  = embed_dim

        self.proj = layers.Conv2D(
            embed_dim, kernel_size=patch_size,
            strides=stride, padding="same"
        )
        self.norm = layers.LayerNormalization(epsilon=1e-6)

        self.patch_weight = None   

    def build(self, input_shape):


        self.patch_weight = self.add_weight(
            name="patch_weight",
            shape=(1, 1, self.embed_dim),   
            initializer="ones",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):

        x = self.proj(x)
        H, W = tf.shape(x)[1], tf.shape(x)[2]


        x_flat = tf.reshape(x, (tf.shape(x)[0], H * W, self.embed_dim))


        x_flat = x_flat * self.patch_weight


        x_norm = self.norm(x_flat)
        return x_norm, H, W


class RelativePositionBias2D(layers.Layer):
    """
    Learn a relative positional bias for each patch pair (i, j).
    B_RPE[i,j,k] = B_height[yi-yj + Sy, k] + B_width[xi-xj + Sx, k]
    """
    def __init__(self, num_heads, max_relative_position=32, **kwargs):
        super().__init__(**kwargs)
        self.num_heads           = num_heads
        self.max_relative_pos    = max_relative_position

        table_size = 2 * max_relative_position - 1
        self.relative_height = self.add_weight(
            name="rel_height", shape=(table_size, num_heads),
            initializer="zeros", trainable=True
        )
        self.relative_width  = self.add_weight(
            name="rel_width",  shape=(table_size, num_heads),
            initializer="zeros", trainable=True
        )

    def call(self, H, W):
        """Returns RPE bias of shape (num_heads, H*W, H*W)."""
        coords_h = tf.range(H)
        coords_w = tf.range(W)

        grid_h, grid_w = tf.meshgrid(coords_h, coords_w, indexing='ij')
        grid_h = tf.reshape(grid_h, (-1,))  
        grid_w = tf.reshape(grid_w, (-1,))


        S = self.max_relative_pos - 1
        rel_h = tf.clip_by_value(
            grid_h[:, None] - grid_h[None, :], -(S), S
        ) + S   
        rel_w = tf.clip_by_value(
            grid_w[:, None] - grid_w[None, :], -(S), S
        ) + S


        bias_h = tf.gather(self.relative_height, rel_h)
        bias_w = tf.gather(self.relative_width,  rel_w)
        bias   = bias_h + bias_w   


        bias = tf.transpose(bias, (2, 0, 1))
        return bias


class HausdorffMultiHeadAttention(layers.Layer):
    """
    Standard MHA + Hausdorff spatial bias + RPE bias.

    Attention score pipeline (paper Eq.4 & 7):
        S        = Q @ K^T / sqrt(d_k)
        S'       = S + B_Hausdorff          (Eq.4)
        S''      = S' + B_RPE               (Eq.7)
        output   = softmax(S'') @ V
    """
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.q_proj = layers.Dense(embed_dim)
        self.k_proj = layers.Dense(embed_dim)
        self.v_proj = layers.Dense(embed_dim)
        self.out    = layers.Dense(embed_dim)

    @staticmethod
    def _hausdorff_bias(H, W):
        """
        Pre-compute Hausdorff bias from Euclidean distances (Eq.3).
        B_ij = -d(pi, pj) / (max(D) + eps)
        Returns (1, 1, N, N) tensor (broadcast over batch & heads).
        """
        coords_h = tf.cast(tf.range(H), tf.float32)
        coords_w = tf.cast(tf.range(W), tf.float32)
        gh, gw   = tf.meshgrid(coords_h, coords_w, indexing='ij')
        gh = tf.reshape(gh, (-1,))  
        gw = tf.reshape(gw, (-1,))


        dh = gh[:, None] - gh[None, :]
        dw = gw[:, None] - gw[None, :]
        D  = tf.sqrt(dh**2 + dw**2)

        eps   = 1e-6
        B_hd  = -D / (tf.reduce_max(D) + eps)   

        return B_hd[None, None, :, :]

    def call(self, x, rpe_bias=None, H=None, W=None):
        B  = tf.shape(x)[0]
        N  = tf.shape(x)[1]
        nh = self.num_heads
        hd = self.head_dim

        Q = tf.reshape(self.q_proj(x), (B, N, nh, hd))
        K = tf.reshape(self.k_proj(x), (B, N, nh, hd))
        V = tf.reshape(self.v_proj(x), (B, N, nh, hd))


        Q = tf.transpose(Q, (0,2,1,3))
        K = tf.transpose(K, (0,2,1,3))
        V = tf.transpose(V, (0,2,1,3))


        S = tf.matmul(Q, K, transpose_b=True) * self.scale   


        if H is not None and W is not None:
            S = S + tf.cast(self._hausdorff_bias(H, W), S.dtype)


        if rpe_bias is not None:

            S = S + tf.cast(rpe_bias[None], S.dtype)


        S = S - tf.reduce_max(S, axis=-1, keepdims=True)
        attn = tf.nn.softmax(S)

        out = tf.matmul(attn, V)               
        out = tf.transpose(out, (0,2,1,3))     
        out = tf.reshape(out, (B, N, nh*hd))
        return self.out(out)


class MixFFN(layers.Layer):
    """
    fc1 → GELU → reshape → DepthwiseConv2D → flatten → LN → fc2
    DepthwiseConv2D adds local spatial context per channel (Eq.5).
    """
    def __init__(self, embed_dim, expand_ratio=4, **kwargs):
        super().__init__(**kwargs)
        exp = embed_dim * expand_ratio
        self.fc1   = layers.Dense(exp)
        self.act   = layers.Activation("gelu")
        self.dwconv = layers.DepthwiseConv2D(
            kernel_size=3, padding="same", use_bias=True
        )
        self.norm  = layers.LayerNormalization(epsilon=1e-6)
        self.fc2   = layers.Dense(embed_dim)

    def call(self, x, H, W):
        u = self.act(self.fc1(x))                           

        B   = tf.shape(u)[0]
        exp = tf.shape(u)[-1]
        u_s = tf.reshape(u, (B, H, W, exp))                 
        u_s = self.dwconv(u_s)                              
        u   = tf.reshape(u_s, (B, H*W, exp))
        u   = self.norm(u)
        return self.fc2(u)


class TransformerEncoderLayer(layers.Layer):
    def __init__(self, embed_dim, num_heads, max_rel_pos=32, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.attn  = HausdorffMultiHeadAttention(embed_dim, num_heads)
        self.ffn   = MixFFN(embed_dim)
        self.rpe   = RelativePositionBias2D(num_heads, max_rel_pos)
        self.drop  = layers.Dropout(0.1)

    def call(self, x, H, W, training=False):
        rpe_bias = self.rpe(H, W)

        x = x + self.drop(
            self.attn(self.norm1(x), rpe_bias=rpe_bias, H=H, W=W),
            training=training
        )

        x = x + self.drop(
            self.ffn(self.norm2(x), H, W),
            training=training
        )
        return x


In [ ]:
class MLSRSSegFormer(tf.keras.Model):
    STAGE_CFG = [
        (7, 4,  32,  1, 2),
        (3, 2,  64,  2, 2),
        (3, 2, 160,  5, 2),
        (3, 2, 256,  8, 2),
    ]

    def __init__(self, num_classes=2, **kwargs):
        super().__init__(**kwargs)

        self.patch_embs = []
        self.enc_blocks = []
        for i, (ps, st, edim, nh, nb) in enumerate(self.STAGE_CFG):
            self.patch_embs.append(PatchEmbedding(ps, st, edim, name=f"pe_{i}"))
            self.enc_blocks.append([
                TransformerEncoderLayer(edim, nh, name=f"enc_{i}_{j}")
                for j in range(nb)
            ])

        target_ch = 128
        self.dec_projs = [
            layers.Conv2D(target_ch, 1, padding="same", name=f"dp_{i}")
            for i in range(len(self.STAGE_CFG))
        ]
        self.dec_ups = [
            layers.UpSampling2D((2**i, 2**i), interpolation="bilinear") if 2**i > 1 else None
            for i in range(len(self.STAGE_CFG))
        ]

        self.dec_conv = layers.Conv2D(128, 3, padding="same", activation="relu")
        self.up1      = layers.UpSampling2D((2,2), interpolation="bilinear")
        self.conv1    = layers.Conv2D(64,  3, padding="same", activation="relu")
        self.up2      = layers.UpSampling2D((2,2), interpolation="bilinear")
        self.conv2    = layers.Conv2D(32,  3, padding="same", activation="relu")
        self.seg_head = layers.Conv2D(num_classes, 1, activation="softmax")

    def call(self, x, training=False):
        feature_maps = []
        for patch_emb, blocks in zip(self.patch_embs, self.enc_blocks):
            tokens, H, W = patch_emb(x)
            for blk in blocks:
                tokens = blk(tokens, H, W, training=training)
            B    = tf.shape(tokens)[0]
            feat = tf.reshape(tokens, (B, H, W, tf.shape(tokens)[-1]))
            feature_maps.append(feat)
            x = feat

        fused = None
        for i, feat in enumerate(feature_maps):
            y = self.dec_projs[i](feat)
            if self.dec_ups[i] is not None:
                y = self.dec_ups[i](y)
            fused = y if fused is None else fused + y

        x = self.dec_conv(fused)
        x = self.up1(x);  x = self.conv1(x)
        x = self.up2(x);  x = self.conv2(x)
        return self.seg_head(x)


model = MLSRSSegFormer(num_classes=NUM_CLASSES)
_ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))  
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=total_loss,
    metrics=["accuracy", dice_coef, iou]
)


In [ ]:
import keras_tuner as kt

def build_model(hp):
    learning_rate = hp.Choice("learning_rate", [1e-3, 5e-4, 1e-4, 5e-5])
    expand_ratio  = hp.Choice("expand_ratio", [2, 4])

    class MLSRSSegFormerTuned(tf.keras.Model):
        STAGE_CFG = [
            (7, 4,  32,  1, 2),
            (3, 2,  64,  2, 2),
            (3, 2, 160,  5, 2),
            (3, 2, 256,  8, 2),
        ]

        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self.patch_embs = []
            self.enc_blocks = []
            for i, (ps, st, edim, nh, nb) in enumerate(self.STAGE_CFG):
                self.patch_embs.append(PatchEmbedding(ps, st, edim, name=f"pe_{i}"))
                self.enc_blocks.append([
                    TransformerEncoderLayer(edim, nh, name=f"enc_{i}_{j}")
                    for j in range(nb)
                ])
            target_ch = 128
            self.dec_projs = [
                layers.Conv2D(target_ch, 1, padding="same", name=f"dp_{i}")
                for i in range(len(self.STAGE_CFG))
            ]
            self.dec_ups = [
                layers.UpSampling2D((2**i, 2**i), interpolation="bilinear")
                if 2**i > 1 else None
                for i in range(len(self.STAGE_CFG))
            ]
            self.dec_conv = layers.Conv2D(128, 3, padding="same", activation="relu")
            self.up1      = layers.UpSampling2D((2,2), interpolation="bilinear")
            self.conv1    = layers.Conv2D(64, 3, padding="same", activation="relu")
            self.up2      = layers.UpSampling2D((2,2), interpolation="bilinear")
            self.conv2    = layers.Conv2D(32, 3, padding="same", activation="relu")
            self.seg_head = layers.Conv2D(NUM_CLASSES, 1, activation="softmax")

        def call(self, x, training=False):
            feature_maps = []
            for patch_emb, blocks in zip(self.patch_embs, self.enc_blocks):
                tokens, H, W = patch_emb(x)
                for blk in blocks:
                    tokens = blk(tokens, H, W, training=training)
                B    = tf.shape(tokens)[0]
                feat = tf.reshape(tokens, (B, H, W, tf.shape(tokens)[-1]))
                feature_maps.append(feat)
                x = feat
            fused = None
            for i, feat in enumerate(feature_maps):
                y = self.dec_projs[i](feat)
                if self.dec_ups[i] is not None:
                    y = self.dec_ups[i](y)
                fused = y if fused is None else fused + y
            x = self.dec_conv(fused)
            x = self.up1(x); x = self.conv1(x)
            x = self.up2(x); x = self.conv2(x)
            return self.seg_head(x)

    model = MLSRSSegFormerTuned()
    _ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss=total_loss,
        metrics=["accuracy", dice_coef, iou]
    )
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective=kt.Objective("val_loss", direction="min"),
    max_trials=3,
    directory="kt_results",
    project_name="mlsrs_tuning"
)

tuner.search_space_summary()

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=1)
    ]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best learning rate:", best_hps.get("learning_rate"))
print("Selected expand ratio (not wired into blocks):", best_hps.get("expand_ratio"))

best_model = tuner.get_best_models(num_models=1)[0]
history_tuned = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                         restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint("mlsrs_tuned_best.h5",
                                           monitor="val_loss", save_best_only=True)
    ]
)

In [ ]:
best_model.load_weights("mlsrs_tuned_best.h5")
logger.info("Loaded")

In [ ]:
import random
import os

os.makedirs("results_tuned", exist_ok=True)

test_imgs   = get_all_files("/kaggle/input/datasets/mehmetkaraman48/breastdm/seg/test/images")
random_imgs = random.sample(test_imgs, 5)

for idx, img_path in enumerate(random_imgs):
    predict_and_save(img_path, best_model, f"results_tuned/overlay_{idx+1}.png")
    logger.info(f"overlay_{idx+1}.png saved")

In [ ]:
def predict_and_save(img_path, model, save_path, img_size=256):
    img  = tf.io.read_file(img_path)
    img  = tf.image.decode_jpeg(img, channels=3)
    orig = img.numpy()
    img  = tf.image.resize(img, (img_size, img_size))
    img  = tf.cast(img, tf.float32) / 255.0
    pred = model.predict(tf.expand_dims(img, 0), verbose=0)
    mask = tf.argmax(pred[0], axis=-1).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig);              axes[0].set_title("Original Image"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Predicted Mask"); axes[1].axis("off")
    overlay = img.numpy().copy()
    overlay[mask == 1] = [1.0, 0.0, 0.0]
    axes[2].imshow(overlay);           axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
import random, os
os.makedirs("results_tuned", exist_ok=True)

test_imgs   = get_all_files("/kaggle/input/datasets/mehmetkaraman48/breastdm/seg/test/images")
random_imgs = random.sample(test_imgs, 5)

for idx, img_path in enumerate(random_imgs):
    predict_and_save(img_path, best_model, f"results_tuned/overlay_{idx+1}.png")
    logger.info(f"overlay_{idx+1}.png saved")

In [ ]:
best_model.load_weights("mlsrs_tuned_best.h5")
logger.info("Best weights loaded")

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("results_tuned", exist_ok=True)
count = 0

for images, masks in val_ds.take(5):
    preds = best_model.predict(images, verbose=0)  
    for i in range(len(images)):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(images[i]);                             axes[0].set_title("Image");          axes[0].axis("off")
        axes[1].imshow(tf.argmax(masks[i], -1), cmap="gray"); axes[1].set_title("True Mask");      axes[1].axis("off")
        axes[2].imshow(tf.argmax(preds[i], -1), cmap="gray"); axes[2].set_title("Predicted Mask"); axes[2].axis("off")
        plt.tight_layout()
        plt.savefig(f"results_tuned/val_comparison_{count+1}.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()
        count += 1
        logger.info(f"val_comparison_{count}.png saved")

print(f"\nTotal {count} images saved")

In [ ]:
print(history_tuned.history.keys())

In [ ]:
y_true_list = []
y_pred_list = []

for images, masks in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true_list.append(masks.numpy())
    y_pred_list.append(preds)

y_true_tuned = np.concatenate(y_true_list, axis=0)
y_pred_tuned = np.concatenate(y_pred_list, axis=0)
print("Ready:", y_true_tuned.shape, y_pred_tuned.shape)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os
from scipy.spatial.distance import directed_hausdorff

os.makedirs("results_tuned", exist_ok=True)


train_dice = history_tuned.history['dice_coef']
val_dice   = history_tuned.history['val_dice_coef']
train_iou  = history_tuned.history['iou']
val_iou    = history_tuned.history['val_iou']
train_loss = history_tuned.history['loss']
val_loss   = history_tuned.history['val_loss']
epochs     = list(range(1, len(train_dice)+1))
best_ep    = val_loss.index(min(val_loss)) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("MLSRS-SegFormer Tuned — Training Results (BreastDM)", fontsize=14, fontweight="bold")

axes[0].plot(epochs, train_dice, 'b-o', label="Train Dice", linewidth=2)
axes[0].plot(epochs, val_dice,   'r-o', label="Val Dice",   linewidth=2)
axes[0].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best (Ep.{best_ep})")
axes[0].set_title("Dice Coefficient"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Dice")
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(0, 1)

axes[1].plot(epochs, train_iou, 'b-o', label="Train IoU", linewidth=2)
axes[1].plot(epochs, val_iou,   'r-o', label="Val IoU",   linewidth=2)
axes[1].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best (Ep.{best_ep})")
axes[1].set_title("IoU Score"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("IoU")
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 1)

axes[2].plot(epochs, train_loss, 'b-o', label="Train Loss", linewidth=2)
axes[2].plot(epochs, val_loss,   'r-o', label="Val Loss",   linewidth=2)
axes[2].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best (Ep.{best_ep})")
axes[2].set_title("Loss"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results_tuned/training_curves_tuned.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("training_curves_tuned.png")


fig2, ax = plt.subplots(figsize=(13, len(epochs)*0.5+1))
ax.axis("off")
columns = ["Epoch","Train Dice","Val Dice","Train IoU","Val IoU","Train Loss","Val Loss"]
rows = [[e, f"{train_dice[i]:.4f}", f"{val_dice[i]:.4f}",
            f"{train_iou[i]:.4f}",  f"{val_iou[i]:.4f}",
            f"{train_loss[i]:.4f}", f"{val_loss[i]:.4f}"]
        for i, e in enumerate(epochs)]
tbl = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
for col in range(len(columns)):
    tbl[0, col].set_facecolor("#2E75B6")
    tbl[0, col].set_text_props(color="white", fontweight="bold")
    tbl[best_ep, col].set_facecolor("#c8f7c5")
ax.set_title("MLSRS-SegFormer Tuned — Training Metrics", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("results_tuned/training_table_tuned.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("training_table_tuned.png")


fig3, axes3 = plt.subplots(1, 2, figsize=(16, 6))
fig3.suptitle("MLSRS-SegFormer Tuned — Additional Plots", fontsize=14, fontweight="bold")

labels  = ["Stage 1\nEncoder","Stage 2\nEncoder","Stage 3\nEncoder","Stage 4\nEncoder","Decoder"]
sizes   = [33098, 125304, 727980, 2005672, 276530]
colors  = ["#534AB7","#1D9E75","#D85A30","#3C3489","#888780"]
wedges, texts, autotexts = axes3[0].pie(
    sizes, labels=labels, colors=colors, explode=(0,0,0,0.05,0),
    autopct="%1.1f%%", startangle=140,
    textprops={"fontsize":10}, wedgeprops={"edgecolor":"white","linewidth":1.5})
for at in autotexts:
    at.set_fontsize(9); at.set_fontweight("bold"); at.set_color("white")
axes3[0].set_title("Parameter Distribution\n(Total: 3,168,578)", fontsize=12, fontweight="bold")

deltas     = [0] + [round(val_dice[i]-val_dice[i-1],4) for i in range(1,len(val_dice))]
bar_colors = ["#1D9E75" if d >= 0 else "#D85A30" for d in deltas]
axes3[1].bar(epochs, deltas, color=bar_colors, edgecolor="white", linewidth=0.8)
axes3[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes3[1].axvline(x=best_ep, color="#534AB7", linewidth=1.5, linestyle="--", label=f"Best (Ep.{best_ep})")
up_patch   = mpatches.Patch(color="#1D9E75", label="Increase")
down_patch = mpatches.Patch(color="#D85A30", label="Decrease")
axes3[1].legend(handles=[up_patch, down_patch,
    plt.Line2D([0],[0], color="#534AB7", linestyle="--")],
    labels=["Increase","Decrease",f"Best (Ep.{best_ep})"], fontsize=9)
axes3[1].set_title("Epoch-to-Epoch Change in Validation Dice", fontsize=12, fontweight="bold")
axes3[1].set_xlabel("Epoch"); axes3[1].set_ylabel("Δ Val Dice")
axes3[1].set_xticks(epochs); axes3[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results_tuned/additional_plots_tuned.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("additional_plots_tuned.png")


def compute_area(mask_binary):
    return np.sum(mask_binary)

def compute_hd95(pred_bin, true_bin):
    pred_pts = np.argwhere(pred_bin)
    true_pts = np.argwhere(true_bin)
    if len(pred_pts) == 0 or len(true_pts) == 0:
        return 0.0
    d1 = directed_hausdorff(pred_pts, true_pts)[0]
    d2 = directed_hausdorff(true_pts, pred_pts)[0]
    return max(d1, d2) * 0.95

true_areas, pred_areas, hd95_vals = [], [], []
print("Computing Bland-Altman statistics...")
for i in range(len(y_true_tuned)):
    true_bin = np.argmax(y_true_tuned[i], axis=-1)
    pred_bin = np.argmax(y_pred_tuned[i], axis=-1)
    true_areas.append(compute_area(true_bin))
    pred_areas.append(compute_area(pred_bin))
    hd95_vals.append(compute_hd95(pred_bin, true_bin))
    if i % 500 == 0:
        print(f"  {i}/{len(y_true_tuned)}")

print("Plotting...")
true_a = np.array(true_areas, dtype=float)
pred_a = np.array(pred_areas, dtype=float)
diff   = pred_a - true_a
mean   = (pred_a + true_a) / 2.0
md     = np.mean(diff); sd = np.std(diff)
upper  = md + 1.96*sd;  lower = md - 1.96*sd

fig4, ax4 = plt.subplots(figsize=(9, 6))
ax4.scatter(mean, diff, alpha=0.35, s=16, color="#1D9E75", edgecolors="none")
ax4.axhline(md,    color="#D85A30", linewidth=1.8, linestyle="--", label=f"Mean diff: {md:.2f} px²")
ax4.axhline(upper, color="#534AB7", linewidth=1.3, linestyle=":", label=f"+1.96 SD: {upper:.2f} px²")
ax4.axhline(lower, color="#534AB7", linewidth=1.3, linestyle=":", label=f"-1.96 SD: {lower:.2f} px²")
ax4.set_title("Bland-Altman: Area — Tuned Model (BreastDM Test)", fontsize=12, fontweight="bold")
ax4.set_xlabel("Mean Area (pixel²)"); ax4.set_ylabel("Difference: Prediction − Reference (pixel²)")
ax4.legend(); ax4.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("results_tuned/bland_altman_tuned.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("bland_altman_tuned.png")


hd95_arr = np.array(hd95_vals)
nonzero  = hd95_arr[hd95_arr > 0]
fig5, ax5 = plt.subplots(figsize=(9, 4))
ax5.axis("off")
tbl_data = [
    ["Mean Legacy Distance",       f"{np.mean(hd95_arr):.3f} px"],
    ["Median Legacy Distance",         f"{np.median(hd95_arr):.3f} px"],
    ["Std HD95",            f"{np.std(hd95_arr):.3f} px"],
    ["Min HD95",            f"{np.min(hd95_arr):.3f} px"],
    ["Max HD95",            f"{np.max(hd95_arr):.3f} px"],
    ["Nonzero Samples", f"{len(nonzero)} / {len(hd95_arr)}"],
]
tbl2 = ax5.table(cellText=tbl_data, colLabels=["Metric","Value"], cellLoc="center", loc="center")
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.3, 2.0)
for col in range(2):
    tbl2[0, col].set_facecolor("#534AB7")
    tbl2[0, col].set_text_props(color="white", fontweight="bold")
ax5.set_title("Legacy Distance Summary — Tuned Model (BreastDM Test)", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("results_tuned/hd95_tuned.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("hd95_tuned.png")

logger.info("All plots saved to results_tuned/!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_dice = history_tuned.history['dice_coef']
val_dice   = history_tuned.history['val_dice_coef']
train_iou  = history_tuned.history['iou']
val_iou    = history_tuned.history['val_iou']
train_loss = history_tuned.history['loss']
val_loss   = history_tuned.history['val_loss']
epochs     = list(range(1, len(train_dice)+1))

best_loss_ep = val_loss.index(min(val_loss)) + 1   
best_dice_ep = val_dice.index(max(val_dice)) + 1   

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("MLSRS-SegFormer Tuned — Training Results (BreastDM)", fontsize=14, fontweight="bold")

for ax, y1, y2, title, ylabel in zip(
    axes,
    [train_dice, train_iou, train_loss],
    [val_dice,   val_iou,   val_loss],
    ["Dice Coefficient", "IoU Score", "Loss"],
    ["Dice", "IoU", "Loss"]
):
    ax.plot(epochs, y1, 'b-o', label=f"Train {ylabel}", linewidth=2)
    ax.plot(epochs, y2, 'r-o', label=f"Val {ylabel}",   linewidth=2)
    ax.axvline(x=best_loss_ep, color='green',  linestyle='--', linewidth=1.5,
               label=f"Best Val Loss (Ep.{best_loss_ep})")
    ax.axvline(x=best_dice_ep, color='orange', linestyle='--', linewidth=1.5,
               label=f"Best Val Dice (Ep.{best_dice_ep})")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    if ylabel != "Loss":
        ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig("results_tuned/training_curves_tuned_v2.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"Saved — Best Val Loss: Epoch {best_loss_ep} (green) | Best Val Dice: Epoch {best_dice_ep} (orange)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_dice = history_tuned.history['dice_coef']
val_dice   = history_tuned.history['val_dice_coef']
train_iou  = history_tuned.history['iou']
val_iou    = history_tuned.history['val_iou']
train_loss = history_tuned.history['loss']
val_loss   = history_tuned.history['val_loss']
epochs     = list(range(1, len(train_dice)+1))

best_loss_ep = val_loss.index(min(val_loss)) + 1
best_dice_ep = val_dice.index(max(val_dice)) + 1

fig, ax = plt.subplots(figsize=(13, len(epochs)*0.5+1.5))
ax.axis("off")

columns = ["Epoch","Train Dice","Val Dice","Train IoU","Val IoU","Train Loss","Val Loss"]
rows = [[e, f"{train_dice[i]:.4f}", f"{val_dice[i]:.4f}",
            f"{train_iou[i]:.4f}",  f"{val_iou[i]:.4f}",
            f"{train_loss[i]:.4f}", f"{val_loss[i]:.4f}"]
        for i, e in enumerate(epochs)]

tbl = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)


for col in range(len(columns)):
    tbl[0, col].set_facecolor("#2E75B6")
    tbl[0, col].set_text_props(color="white", fontweight="bold")


for col in range(len(columns)):
    tbl[best_loss_ep, col].set_facecolor("#c8f7c5")  


for col in range(len(columns)):
    tbl[best_dice_ep, col].set_facecolor("#FFE0A0")  


if best_loss_ep == best_dice_ep:
    for col in range(len(columns)):
        tbl[best_loss_ep, col].set_facecolor("#B8F0B0")

ax.set_title(
    f"MLSRS-SegFormer Tuned — Training Metrics\n"
    f"Green: Best Val Loss (Epoch {best_loss_ep}) | Orange: Best Val Dice (Epoch {best_dice_ep})",
    fontsize=12, fontweight="bold", pad=20
)

plt.tight_layout()
plt.savefig("results_tuned/training_table_tuned_v2.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"Saved — Best Val Loss: Epoch {best_loss_ep} (green) | Best Val Dice: Epoch {best_dice_ep} (orange)")

In [ ]:

plt.savefig("results/training_curves_hd.png", dpi=200, bbox_inches="tight")
plt.savefig("results/training_table_hd.png", dpi=200, bbox_inches="tight")
plt.savefig("results_tuned/training_curves_tuned_v2_hd.png", dpi=200, bbox_inches="tight")
plt.savefig("results_tuned/training_table_tuned_v2_hd.png", dpi=200, bbox_inches="tight")

In [ ]:
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt


train_dice = history_tuned.history['dice_coef']
val_dice   = history_tuned.history['val_dice_coef']
train_iou  = history_tuned.history['iou']
val_iou    = history_tuned.history['val_iou']
train_loss = history_tuned.history['loss']
val_loss   = history_tuned.history['val_loss']
epochs     = list(range(1, len(train_dice)+1))

best_loss_ep = val_loss.index(min(val_loss)) + 1
best_dice_ep = val_dice.index(max(val_dice)) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("MLSRS-SegFormer Tuned — Training Results (BreastDM)", fontsize=14, fontweight="bold")

for ax, y1, y2, title, ylabel in zip(
    axes,
    [train_dice, train_iou, train_loss],
    [val_dice,   val_iou,   val_loss],
    ["Dice Coefficient", "IoU Score", "Loss"],
    ["Dice", "IoU", "Loss"]
):
    ax.plot(epochs, y1, 'b-o', label=f"Train {ylabel}", linewidth=2)
    ax.plot(epochs, y2, 'r-o', label=f"Val {ylabel}", linewidth=2)
    ax.axvline(x=best_loss_ep, color='green', linestyle='--', linewidth=1.5, label=f"Best Val Loss (Ep.{best_loss_ep})")
    ax.axvline(x=best_dice_ep, color='orange', linestyle='--', linewidth=1.5, label=f"Best Val Dice (Ep.{best_dice_ep})")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    if ylabel != "Loss":
        ax.set_ylim(0, 1)

plt.tight_layout()
fig.savefig("results_tuned/curves_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("curves_hd.png")

In [ ]:
fig, ax = plt.subplots(figsize=(13, len(epochs)*0.5+1.5))
ax.axis("off")

columns = ["Epoch","Train Dice","Val Dice","Train IoU","Val IoU","Train Loss","Val Loss"]
rows = [[e, f"{train_dice[i]:.4f}", f"{val_dice[i]:.4f}",
            f"{train_iou[i]:.4f}",  f"{val_iou[i]:.4f}",
            f"{train_loss[i]:.4f}", f"{val_loss[i]:.4f}"]
        for i, e in enumerate(epochs)]

tbl = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)

for col in range(len(columns)):
    tbl[0, col].set_facecolor("#2E75B6")
    tbl[0, col].set_text_props(color="white", fontweight="bold")

for col in range(len(columns)):
    tbl[best_loss_ep, col].set_facecolor("#c8f7c5")  
for col in range(len(columns)):
    tbl[best_dice_ep, col].set_facecolor("#FFE0A0")  

ax.set_title(
    f"MLSRS-SegFormer Tuned — Training Metrics\n"
    f"Green: Best Val Loss (Epoch {best_loss_ep}) | Orange: Best Val Dice (Epoch {best_dice_ep})",
    fontsize=12, fontweight="bold", pad=20
)

plt.tight_layout()
fig.savefig("results_tuned/table_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("table_hd.png")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

def compute_area(mask_binary): return np.sum(mask_binary)
def compute_hd95(pred_bin, true_bin):
    p = np.argwhere(pred_bin); t = np.argwhere(true_bin)
    if len(p)==0 or len(t)==0: return 0.0
    return max(directed_hausdorff(p,t)[0], directed_hausdorff(t,p)[0]) * 0.95

true_areas2, pred_areas2, hd95_vals2 = [], [], []
print("Computing...")
for i in range(len(y_true_tuned)):
    tb = np.argmax(y_true_tuned[i], axis=-1)
    pb = np.argmax(y_pred_tuned[i], axis=-1)
    true_areas2.append(compute_area(tb))
    pred_areas2.append(compute_area(pb))
    hd95_vals2.append(compute_hd95(pb, tb))
    if i % 500 == 0: print(f"  {i}/{len(y_true_tuned)}")
print("Done")


ta2 = np.array(true_areas2, dtype=float)
pa2 = np.array(pred_areas2, dtype=float)
diff2 = pa2-ta2; mean2 = (pa2+ta2)/2
md2 = np.mean(diff2); sd2 = np.std(diff2)
upper2 = md2+1.96*sd2; lower2 = md2-1.96*sd2

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(mean2, diff2, alpha=0.35, s=16, color="#1D9E75", edgecolors="none")
ax.axhline(md2,    color="#D85A30", linewidth=1.8, linestyle="--", label=f"Mean diff: {md2:.2f} px²")
ax.axhline(upper2, color="#534AB7", linewidth=1.3, linestyle=":", label=f"+1.96 SD: {upper2:.2f} px²")
ax.axhline(lower2, color="#534AB7", linewidth=1.3, linestyle=":", label=f"-1.96 SD: {lower2:.2f} px²")
ax.set_title("Bland-Altman: Area — Model 2 Tuned (BreastDM Test)", fontsize=12, fontweight="bold")
ax.set_xlabel("Mean Area (pixel²)"); ax.set_ylabel("Difference: Prediction − Reference (pixel²)")
ax.legend(); ax.grid(True, alpha=0.25)
plt.tight_layout()
fig.savefig("results_tuned/bland_altman_m2_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("bland_altman_m2_hd.png")


hd95_arr2 = np.array(hd95_vals2)
nonzero2  = hd95_arr2[hd95_arr2 > 0]
fig, ax = plt.subplots(figsize=(9, 4))
ax.axis("off")
tbl_data2 = [
    ["Mean Legacy Distance", f"{np.mean(hd95_arr2):.3f} px"],
    ["Median Legacy Distance",   f"{np.median(hd95_arr2):.3f} px"],
    ["Std HD95",      f"{np.std(hd95_arr2):.3f} px"],
    ["Min HD95",      f"{np.min(hd95_arr2):.3f} px"],
    ["Max HD95",      f"{np.max(hd95_arr2):.3f} px"],
    ["Nonzero", f"{len(nonzero2)} / {len(hd95_arr2)}"],
]
tbl3 = ax.table(cellText=tbl_data2, colLabels=["Metric","Value"], cellLoc="center", loc="center")
tbl3.auto_set_font_size(False); tbl3.set_fontsize(11); tbl3.scale(1.3, 2.0)
for col in range(2):
    tbl3[0, col].set_facecolor("#534AB7")
    tbl3[0, col].set_text_props(color="white", fontweight="bold")
ax.set_title("Legacy Distance Summary — Model 2 Tuned (BreastDM Test)", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
fig.savefig("results_tuned/hd95_m2_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("hd95_m2_hd.png")

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=15,
        min_lr=1e-15, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=35,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "mlsrs_segformer_best.h5",
        monitor="val_loss", save_best_only=True, verbose=1
    )
]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7,
    callbacks=callbacks
)


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

def predict_and_overlay(img_path, model, img_size=256):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    orig = img.numpy()

    img = tf.image.resize(img, (img_size, img_size))
    img = tf.cast(img, tf.float32) / 255.0
    img_input = tf.expand_dims(img, 0)

    pred = model.predict(img_input, verbose=0)
    mask = tf.argmax(pred[0], axis=-1).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(orig)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Predicted Mask")
    axes[1].axis("off")

    overlay = img.numpy().copy()
    overlay[mask == 1] = [1.0, 0.0, 0.0]
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()

In [ ]:
import random


test_imgs = get_all_files("/kaggle/input/datasets/mehmetkaraman48/breastdm/seg/test/images")
random_imgs = random.sample(test_imgs, 3)

for img_path in random_imgs:
    print(img_path)
    predict_and_overlay(img_path, model)

In [ ]:
import os

ROOT = "/kaggle/input/datasets/mehmetkaraman48/breastdm"


for split in ["train", "val", "test"]:
    img_dir  = os.path.join(ROOT, "seg", split, "images")
    mask_dir = os.path.join(ROOT, "seg", split, "labels")

    imgs  = get_all_files(img_dir)
    masks = get_all_files(mask_dir)

    print(f"{split}: {len(imgs)} img, {len(masks)} mask")
    if imgs:
        print(f"  example: {imgs[0]}")

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

count = 0
for images, masks in val_ds.take(5):    
    preds = model.predict(images, verbose=0)
    for i in range(len(images)):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(images[i]);                             axes[0].set_title("Image");          axes[0].axis("off")
        axes[1].imshow(tf.argmax(masks[i], -1), cmap="gray"); axes[1].set_title("True Mask");      axes[1].axis("off")
        axes[2].imshow(tf.argmax(preds[i], -1), cmap="gray"); axes[2].set_title("Predicted Mask"); axes[2].axis("off")
        plt.tight_layout()
        plt.savefig(f"results/val_comparison_{count+1}.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()
        count += 1
        logger.info(f"val_comparison_{count}.png saved")

print(f"\nTotal {count} images saved")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
history_data = {
    'dice_coef':     [0.6455, 0.6815, 0.7124, 0.7372, 0.7528, 0.7694, 0.7781, 0.7808, 0.7965, 0.8061, 0.8159, 0.8269, 0.8363, 0.8076, 0.8178, 0.8298, 0.8372, 0.8449, 0.8501, 0.8555],
    'val_dice_coef': [0.3142, 0.3603, 0.4246, 0.4273, 0.4466, 0.5015, 0.4743, 0.4719, 0.5447, 0.5117, 0.5451, 0.5510, 0.5361, 0.5462, 0.5363, 0.5658, 0.5080, 0.5507, 0.5075, 0.5441],
    'iou':           [0.5382, 0.5779, 0.6109, 0.6366, 0.6541, 0.6709, 0.6804, 0.6843, 0.7021, 0.7120, 0.7221, 0.7345, 0.7455, 0.7124, 0.7246, 0.7367, 0.7459, 0.7544, 0.7614, 0.7678],
    'val_iou':       [0.2475, 0.2837, 0.3316, 0.3397, 0.3560, 0.4005, 0.3754, 0.3779, 0.4410, 0.4122, 0.4455, 0.4478, 0.4321, 0.4427, 0.4325, 0.4642, 0.4042, 0.4488, 0.4124, 0.4429],
    'loss':          [0.0044, 0.0039, 0.0035, 0.0032, 0.0031, 0.0029, 0.0028, 0.0028, 0.0025, 0.0024, 0.0024, 0.0022, 0.0021, 0.0024, 0.0023, 0.0022, 0.0021, 0.0020, 0.0020, 0.0019],
    'val_loss':      [0.0139, 0.0110, 0.0113, 0.0119, 0.0116, 0.0133, 0.0112, 0.0123, 0.0094, 0.0136, 0.0100, 0.0107, 0.0114, 0.0086, 0.0097, 0.0084, 0.0135, 0.0110, 0.0123, 0.0122],
}

class FakeHistory:
    def __init__(self, data):
        self.history = data

history = FakeHistory(history_data)
logger.info("History updated: 20 epochs")

class FakeHistory:
    def __init__(self, data):
        self.history = data

history = FakeHistory(history_data)

train_dice  = history.history['dice_coef']
val_dice    = history.history['val_dice_coef']
train_iou   = history.history['iou']
val_iou     = history.history['val_iou']
train_loss  = history.history['loss']
val_loss    = history.history['val_loss']
epochs      = list(range(1, len(train_dice) + 1))

os.makedirs("resultss", exist_ok=True)


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("MLSRS-SegFormer Training Results (BreastDM)", fontsize=14, fontweight="bold")

best_ep = val_loss.index(min(val_loss)) + 1

axes[0].plot(epochs, train_dice, 'b-o', label="Train Dice", linewidth=2)
axes[0].plot(epochs, val_dice,   'r-o', label="Val Dice",   linewidth=2)
axes[0].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best Model (Ep.{best_ep})")
axes[0].set_title("Dice Coefficient")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Dice")
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(0, 1)

axes[1].plot(epochs, train_iou, 'b-o', label="Train IoU", linewidth=2)
axes[1].plot(epochs, val_iou,   'r-o', label="Val IoU",   linewidth=2)
axes[1].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best Model (Ep.{best_ep})")
axes[1].set_title("IoU Score")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("IoU")
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 1)

axes[2].plot(epochs, train_loss, 'b-o', label="Train Loss", linewidth=2)
axes[2].plot(epochs, val_loss,   'r-o', label="Val Loss",   linewidth=2)
axes[2].axvline(x=best_ep, color='green', linestyle='--', alpha=0.7, label=f"Best Model (Ep.{best_ep})")
axes[2].set_title("Loss")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("resultss/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("training_curves.png saved")


fig2, ax = plt.subplots(figsize=(12, len(epochs) * 0.5 + 1))
ax.axis("off")

columns = ["Epoch", "Train Dice", "Val Dice", "Train IoU", "Val IoU", "Train Loss", "Val Loss"]
rows = [[
    e,
    f"{train_dice[i]:.4f}", f"{val_dice[i]:.4f}",
    f"{train_iou[i]:.4f}",  f"{val_iou[i]:.4f}",
    f"{train_loss[i]:.4f}", f"{val_loss[i]:.4f}",
] for i, e in enumerate(epochs)]

table = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

for col in range(len(columns)):
    table[0, col].set_facecolor("#2c3e50")
    table[0, col].set_text_props(color="white", fontweight="bold")
    table[best_ep, col].set_facecolor("#c8f7c5")  

ax.set_title("MLSRS-SegFormer Training Metrics (BreastDM Dataset)",
             fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("resultss/training_table.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("training_table.png saved")


import random

def predict_and_save(img_path, model, save_path, img_size=256):
    img  = tf.io.read_file(img_path)
    img  = tf.image.decode_jpeg(img, channels=3)
    orig = img.numpy()
    img  = tf.image.resize(img, (img_size, img_size))
    img  = tf.cast(img, tf.float32) / 255.0
    pred = model.predict(tf.expand_dims(img, 0), verbose=0)
    mask = tf.argmax(pred[0], axis=-1).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig);                              axes[0].set_title("Original Image"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray");                 axes[1].set_title("Predicted Mask"); axes[1].axis("off")
    overlay = img.numpy().copy()
    overlay[mask == 1] = [1.0, 0.0, 0.0]
    axes[2].imshow(overlay);                           axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

test_imgs   = get_all_files("/kaggle/input/datasets/mehmetkaraman48/breastdm/seg/test/images")
random_imgs = random.sample(test_imgs, 5)

for idx, img_path in enumerate(random_imgs):
    save_path = f"results/overlay_{idx+1}.png"
    predict_and_save(img_path, model, save_path)
    logger.info(f"overlay_{idx+1}.png saved")

logger.info("All images saved to results/")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("MLSRS-SegFormer — Additional Plots", fontsize=14, fontweight="bold")


labels  = ["Stage 1\nEncoder", "Stage 2\nEncoder", "Stage 3\nEncoder",
           "Stage 4\nEncoder", "Decoder"]
sizes   = [33098, 125304, 727980, 2005672, 276530]
colors  = ["#534AB7", "#1D9E75", "#D85A30", "#3C3489", "#888780"]
explode = (0, 0, 0, 0.05, 0)

wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct="%1.1f%%", startangle=140,
    textprops={"fontsize": 10},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight("bold")
    at.set_color("white")

axes[0].set_title("Parameter Distribution\n(Total: 3,168,578)", fontsize=12, fontweight="bold")


epochs    = list(range(1, 21))
val_dice  = [0.3142, 0.3603, 0.4246, 0.4273, 0.4466,
             0.5015, 0.4743, 0.4719, 0.5447, 0.5117,
             0.5451, 0.5510, 0.5361, 0.5462, 0.5363,
             0.5658, 0.5080, 0.5507, 0.5075, 0.5441]

deltas = [0] + [round(val_dice[i] - val_dice[i-1], 4) for i in range(1, len(val_dice))]
bar_colors = ["#1D9E75" if d >= 0 else "#D85A30" for d in deltas]

bars = axes[1].bar(epochs, deltas, color=bar_colors, edgecolor="white", linewidth=0.8)
axes[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[1].axvline(x=16, color="#534AB7", linewidth=1.5, linestyle="--", label="Best Model (Ep.16)")

axes[1].set_title("Epoch-to-Epoch Change in Validation Dice", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Δ Val Dice")
axes[1].set_xticks(epochs)
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].legend(fontsize=9)

up_patch   = mpatches.Patch(color="#1D9E75", label="Increase")
down_patch = mpatches.Patch(color="#D85A30", label="Decrease")
axes[1].legend(handles=[up_patch, down_patch, plt.Line2D([0],[0], color="#534AB7", linestyle="--")],
               labels=["Increase", "Decrease", "Best Model (Ep.16)"], fontsize=9)

plt.tight_layout()
plt.savefig("results/additional_plots.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("additional_plots.png saved")

In [ ]:

import numpy as np

y_true_list = []
y_pred_list = []

for images, masks in test_ds:
    preds = model.predict(images, verbose=0)
    y_true_list.append(masks.numpy())
    y_pred_list.append(preds)

y_true = np.concatenate(y_true_list, axis=0)
y_pred = np.concatenate(y_pred_list, axis=0)
print("Test predictions ready:", y_true.shape, y_pred.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

def compute_area(mask_binary):
    return np.sum(mask_binary)

def compute_hd95(pred_bin, true_bin):
    pred_pts = np.argwhere(pred_bin)
    true_pts = np.argwhere(true_bin)
    if len(pred_pts) == 0 or len(true_pts) == 0:
        return 0.0
    d1 = directed_hausdorff(pred_pts, true_pts)[0]
    d2 = directed_hausdorff(true_pts, pred_pts)[0]
    return max(d1, d2) * 0.95

def bland_altman_plot(ax, true_vals, pred_vals, title, unit=""):
    true_vals = np.array(true_vals, dtype=float)
    pred_vals = np.array(pred_vals, dtype=float)
    diff      = pred_vals - true_vals
    mean      = (pred_vals + true_vals) / 2.0
    md        = np.mean(diff)
    sd        = np.std(diff)
    upper     = md + 1.96 * sd
    lower     = md - 1.96 * sd

    ax.scatter(mean, diff, alpha=0.4, s=18, color="#1D9E75", edgecolors="none")
    ax.axhline(md,    color="#D85A30", linewidth=1.5, linestyle="--", label=f"Mean diff: {md:.3f}")
    ax.axhline(upper, color="#534AB7", linewidth=1.2, linestyle=":",  label=f"+1.96 SD: {upper:.3f}")
    ax.axhline(lower, color="#534AB7", linewidth=1.2, linestyle=":",  label=f"-1.96 SD: {lower:.3f}")

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(f"Mean ({unit})", fontsize=10)
    ax.set_ylabel(f"Difference (Prediction - Reference) ({unit})", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)


true_areas, pred_areas = [], []
true_hd95s, pred_hd95s = [], []

print("Computing... (this may take a while)")
for i in range(len(y_true)):
    true_bin = np.argmax(y_true[i], axis=-1)
    pred_bin = np.argmax(y_pred[i], axis=-1)

    true_areas.append(compute_area(true_bin))
    pred_areas.append(compute_area(pred_bin))
    true_hd95s.append(compute_hd95(true_bin, true_bin))
    pred_hd95s.append(compute_hd95(pred_bin, true_bin))

    if i % 500 == 0:
        print(f"  {i}/{len(y_true)}")

print("Computation finished; plotting...")


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("MLSRS-SegFormer — Bland-Altman Analysis (BreastDM Test Set)",
             fontsize=13, fontweight="bold")

bland_altman_plot(axes[0], true_areas, pred_areas,
                  "Bland-Altman: Area", unit="pixel²")
bland_altman_plot(axes[1], true_hd95s, pred_hd95s,
                  "Bland-Altman: HD95", unit="pixel")

plt.tight_layout()
plt.savefig("results/bland_altman.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("bland_altman.png saved")

In [ ]:

true_hd95s, pred_hd95s = [], []

for i in range(len(y_true)):
    true_bin = np.argmax(y_true[i], axis=-1)
    pred_bin = np.argmax(y_pred[i], axis=-1)


    true_pts = np.argwhere(true_bin)
    pred_pts = np.argwhere(pred_bin)

    if len(true_pts) == 0 or len(pred_pts) == 0:
        true_hd95s.append(0.0)
        pred_hd95s.append(0.0)
        continue


    hd = compute_hd95(pred_bin, true_bin)
    true_hd95s.append(0.0)    
    pred_hd95s.append(hd)     

    if i % 500 == 0:
        print(f"  {i}/{len(y_true)}")

print("Done")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

def compute_area(mask_binary):
    return np.sum(mask_binary)

def compute_hd95(pred_bin, true_bin):
    pred_pts = np.argwhere(pred_bin)
    true_pts = np.argwhere(true_bin)
    if len(pred_pts) == 0 or len(true_pts) == 0:
        return 0.0
    d1 = directed_hausdorff(pred_pts, true_pts)[0]
    d2 = directed_hausdorff(true_pts, pred_pts)[0]
    return max(d1, d2) * 0.95


true_areas, pred_areas, hd95_vals = [], [], []

print("Computing...")
for i in range(len(y_true)):
    true_bin = np.argmax(y_true[i], axis=-1)
    pred_bin = np.argmax(y_pred[i], axis=-1)
    true_areas.append(compute_area(true_bin))
    pred_areas.append(compute_area(pred_bin))
    hd95_vals.append(compute_hd95(pred_bin, true_bin))
    if i % 500 == 0:
        print(f"  {i}/{len(y_true)}")

print("Done")


true_a = np.array(true_areas, dtype=float)
pred_a = np.array(pred_areas, dtype=float)
diff   = pred_a - true_a
mean   = (pred_a + true_a) / 2.0
md     = np.mean(diff)
sd     = np.std(diff)
upper  = md + 1.96 * sd
lower  = md - 1.96 * sd

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(mean, diff, alpha=0.35, s=16, color="#1D9E75", edgecolors="none")
ax.axhline(md,    color="#D85A30", linewidth=1.8, linestyle="--",
           label=f"Mean diff: {md:.2f} px²")
ax.axhline(upper, color="#534AB7", linewidth=1.3, linestyle=":",
           label=f"+1.96 SD: {upper:.2f} px²")
ax.axhline(lower, color="#534AB7", linewidth=1.3, linestyle=":",
           label=f"-1.96 SD: {lower:.2f} px²")
ax.set_title("Bland-Altman: Area — MLSRS-SegFormer (BreastDM Test Set)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Mean Area (pixel²)", fontsize=11)
ax.set_ylabel("Difference: Prediction − Reference (pixel²)", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("results/bland_altman_area.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("bland_altman_area.png saved")


hd95_arr  = np.array(hd95_vals)
nonzero   = hd95_arr[hd95_arr > 0]

fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.axis("off")
tbl_data = [
    ["Mean Legacy Distance",        f"{np.mean(hd95_arr):.3f} px"],
    ["Median Legacy Distance",          f"{np.median(hd95_arr):.3f} px"],
    ["Std HD95",             f"{np.std(hd95_arr):.3f} px"],
    ["Min HD95",             f"{np.min(hd95_arr):.3f} px"],
    ["Max HD95",             f"{np.max(hd95_arr):.3f} px"],
    ["Nonzero Samples",  f"{len(nonzero)} / {len(hd95_arr)}"],
]
tbl = ax2.table(cellText=tbl_data, colLabels=["Metric", "Value"],
                cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 2.0)
for col in range(2):
    tbl[0, col].set_facecolor("#534AB7")
    tbl[0, col].set_text_props(color="white", fontweight="bold")
ax2.set_title("Legacy Distance Summary Statistics — MLSRS-SegFormer (BreastDM Test Set)",
              fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("results/hd95_summary.png", dpi=150, bbox_inches="tight")
plt.show()
logger.info("hd95_summary.png saved")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

def compute_area(mask_binary): return np.sum(mask_binary)
def compute_hd95(pred_bin, true_bin):
    p = np.argwhere(pred_bin); t = np.argwhere(true_bin)
    if len(p)==0 or len(t)==0: return 0.0
    return max(directed_hausdorff(p,t)[0], directed_hausdorff(t,p)[0]) * 0.95


true_areas, pred_areas, hd95_vals = [], [], []
for i in range(len(y_true)):
    tb = np.argmax(y_true[i], axis=-1)
    pb = np.argmax(y_pred[i], axis=-1)
    true_areas.append(compute_area(tb))
    pred_areas.append(compute_area(pb))
    hd95_vals.append(compute_hd95(pb, tb))
    if i % 500 == 0: print(f"  {i}/{len(y_true)}")


ta = np.array(true_areas, dtype=float)
pa = np.array(pred_areas, dtype=float)
diff = pa - ta; mean = (pa+ta)/2
md = np.mean(diff); sd = np.std(diff)
upper = md+1.96*sd; lower = md-1.96*sd

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(mean, diff, alpha=0.35, s=16, color="#1D9E75", edgecolors="none")
ax.axhline(md,    color="#D85A30", linewidth=1.8, linestyle="--", label=f"Mean diff: {md:.2f} px²")
ax.axhline(upper, color="#534AB7", linewidth=1.3, linestyle=":", label=f"+1.96 SD: {upper:.2f} px²")
ax.axhline(lower, color="#534AB7", linewidth=1.3, linestyle=":", label=f"-1.96 SD: {lower:.2f} px²")
ax.set_title("Bland-Altman: Area — Model 1 (BreastDM Test)", fontsize=12, fontweight="bold")
ax.set_xlabel("Mean Area (pixel²)"); ax.set_ylabel("Difference: Prediction − Reference (pixel²)")
ax.legend(); ax.grid(True, alpha=0.25)
plt.tight_layout()
fig.savefig("results/bland_altman_m1_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("bland_altman_m1_hd.png")


hd95_arr = np.array(hd95_vals)
nonzero  = hd95_arr[hd95_arr > 0]
fig, ax = plt.subplots(figsize=(9, 4))
ax.axis("off")
tbl_data = [
    ["Mean Legacy Distance", f"{np.mean(hd95_arr):.3f} px"],
    ["Median Legacy Distance",   f"{np.median(hd95_arr):.3f} px"],
    ["Std HD95",      f"{np.std(hd95_arr):.3f} px"],
    ["Min HD95",      f"{np.min(hd95_arr):.3f} px"],
    ["Max HD95",      f"{np.max(hd95_arr):.3f} px"],
    ["Nonzero", f"{len(nonzero)} / {len(hd95_arr)}"],
]
tbl2 = ax.table(cellText=tbl_data, colLabels=["Metric","Value"], cellLoc="center", loc="center")
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.3, 2.0)
for col in range(2):
    tbl2[0, col].set_facecolor("#534AB7")
    tbl2[0, col].set_text_props(color="white", fontweight="bold")
ax.set_title("Legacy Distance Summary — Model 1 (BreastDM Test)", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
fig.savefig("results/hd95_m1_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("hd95_m1_hd.png")

In [ ]:

history_data = {
    'dice_coef':     [0.6455, 0.6815, 0.7124, 0.7372, 0.7528, 0.7694, 0.7781, 0.7808, 0.7965, 0.8061, 0.8159, 0.8269, 0.8363, 0.8076, 0.8178, 0.8298, 0.8372, 0.8449, 0.8501, 0.8555],
    'val_dice_coef': [0.3142, 0.3603, 0.4246, 0.4273, 0.4466, 0.5015, 0.4743, 0.4719, 0.5447, 0.5117, 0.5451, 0.5510, 0.5361, 0.5462, 0.5363, 0.5658, 0.5080, 0.5507, 0.5075, 0.5441],
    'iou':           [0.5382, 0.5779, 0.6109, 0.6366, 0.6541, 0.6709, 0.6804, 0.6843, 0.7021, 0.7120, 0.7221, 0.7345, 0.7455, 0.7124, 0.7246, 0.7367, 0.7459, 0.7544, 0.7614, 0.7678],
    'val_iou':       [0.2475, 0.2837, 0.3316, 0.3397, 0.3560, 0.4005, 0.3754, 0.3779, 0.4410, 0.4122, 0.4455, 0.4478, 0.4321, 0.4427, 0.4325, 0.4642, 0.4042, 0.4488, 0.4124, 0.4429],
    'loss':          [0.0044, 0.0039, 0.0035, 0.0032, 0.0031, 0.0029, 0.0028, 0.0028, 0.0025, 0.0024, 0.0024, 0.0022, 0.0021, 0.0024, 0.0023, 0.0022, 0.0021, 0.0020, 0.0020, 0.0019],
    'val_loss':      [0.0139, 0.0110, 0.0113, 0.0119, 0.0116, 0.0133, 0.0112, 0.0123, 0.0094, 0.0136, 0.0100, 0.0107, 0.0114, 0.0086, 0.0097, 0.0084, 0.0135, 0.0110, 0.0123, 0.0122],
}

train_dice = history_data['dice_coef']
val_dice   = history_data['val_dice_coef']
train_iou  = history_data['iou']
val_iou    = history_data['val_iou']
train_loss = history_data['loss']
val_loss   = history_data['val_loss']
epochs     = list(range(1, len(train_dice)+1))

best_loss_ep = val_loss.index(min(val_loss)) + 1
best_dice_ep = val_dice.index(max(val_dice)) + 1


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("MLSRS-SegFormer — Training Results (BreastDM)", fontsize=14, fontweight="bold")

for ax, y1, y2, title, ylabel in zip(
    axes,
    [train_dice, train_iou, train_loss],
    [val_dice,   val_iou,   val_loss],
    ["Dice Coefficient", "IoU Score", "Loss"],
    ["Dice", "IoU", "Loss"]
):
    ax.plot(epochs, y1, 'b-o', label=f"Train {ylabel}", linewidth=2)
    ax.plot(epochs, y2, 'r-o', label=f"Val {ylabel}", linewidth=2)
    ax.axvline(x=best_loss_ep, color='green', linestyle='--', linewidth=1.5, label=f"Best Val Loss (Ep.{best_loss_ep})")
    ax.axvline(x=best_dice_ep, color='orange', linestyle='--', linewidth=1.5, label=f"Best Val Dice (Ep.{best_dice_ep})")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    if ylabel != "Loss":
        ax.set_ylim(0, 1)

plt.tight_layout()
fig.savefig("results/curves_m1_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("curves_m1_hd.png")


fig, ax = plt.subplots(figsize=(13, len(epochs)*0.5+1.5))
ax.axis("off")
columns = ["Epoch","Train Dice","Val Dice","Train IoU","Val IoU","Train Loss","Val Loss"]
rows = [[e, f"{train_dice[i]:.4f}", f"{val_dice[i]:.4f}",
            f"{train_iou[i]:.4f}",  f"{val_iou[i]:.4f}",
            f"{train_loss[i]:.4f}", f"{val_loss[i]:.4f}"]
        for i, e in enumerate(epochs)]
tbl = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
for col in range(len(columns)):
    tbl[0, col].set_facecolor("#2E75B6")
    tbl[0, col].set_text_props(color="white", fontweight="bold")
for col in range(len(columns)):
    tbl[best_loss_ep, col].set_facecolor("#c8f7c5")
for col in range(len(columns)):
    tbl[best_dice_ep, col].set_facecolor("#FFE0A0")
ax.set_title(
    f"MLSRS-SegFormer — Training Metrics\n"
    f"Green: Best Val Loss (Epoch {best_loss_ep}) | Orange: Best Val Dice (Epoch {best_dice_ep})",
    fontsize=12, fontweight="bold", pad=20
)
plt.tight_layout()
fig.savefig("results/table_m1_hd.png", dpi=150, bbox_inches="tight", facecolor='white')
plt.close()
logger.info("table_m1_hd.png")